# Phase 2: Feature Engineering (DSP) — Synchronized

**Objective**: Transform raw vibration segments into a high-dimensional feature vector for classification.
This notebook is **synchronized with `scripts/feature_engineering.py`**.

### Key design decisions (production):
- **Data source**: Only `12k_drive_end` + `normal` folders (consistent 12kHz Drive End sensor)
- **Window**: 2048 samples, stride 512 (75% overlap)
- **Features**: 10 time-domain + 10 frequency-domain + 8 WPD energy bands = 28 total
- **Balancing**: undersample majority / upsample minority → **4,000 samples per class × 4 = 16,000 total**
- **Label encoding**: alphabetical → B=0, IR=1, Normal=2, OR=3

### Output:
- `../data/processed/cwru_features.parquet`

In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.stats
from scipy.fft import fft
import pywt
from tqdm.notebook import tqdm

INTERIM_DIR         = "../data/interim"
PROCESSED_DATA_PATH = "../data/processed"
WINDOW_SIZE         = 2048
STRIDE              = 512

# ── Only these two source categories (consistent 12kHz Drive End)
ALLOWED_CATEGORIES = {"12k_drive_end", "normal"}

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
print("Feature Engineering Setup Complete.")

## 1. Feature Extraction Functions

In [ ]:
def extract_time_features(window):
    """10 time-domain statistical features."""
    abs_w = np.abs(window)
    rms   = np.sqrt(np.mean(window**2))
    peak  = np.max(abs_w)
    mean_abs = np.mean(abs_w)
    return {
        'mean':          np.mean(window),
        'std':           np.std(window),
        'max':           peak,
        'min':           np.min(window),
        'rms':           rms,
        'kurtosis':      scipy.stats.kurtosis(window),
        'skewness':      scipy.stats.skew(window),
        'peak_to_peak':  peak - np.min(window),
        'crest_factor':  peak / rms        if rms      != 0 else 0,
        'shape_factor':  rms / mean_abs    if mean_abs != 0 else 0,
    }

def extract_freq_features(window, fs=12000):
    """10 frequency-domain features from FFT."""
    n = len(window)
    coeffs = fft(window)
    psd    = np.abs(coeffs[:n//2])**2 / n
    freqs  = np.linspace(0, fs/2, n//2)
    total  = np.sum(psd)
    centroid = np.sum(freqs * psd) / total if total > 0 else 0
    spread   = np.sqrt(np.sum((freqs - centroid)**2 * psd) / total) if total > 0 else 0
    cum      = np.cumsum(psd)
    rolloff  = freqs[np.searchsorted(cum, 0.85 * total)] if total > 0 else 0
    # Spectral flatness (Wiener entropy)
    psd_safe = np.where(psd > 0, psd, 1e-12)
    flatness = np.exp(np.mean(np.log(psd_safe))) / np.mean(psd) if np.mean(psd) > 0 else 0
    # 6 band energies (split spectrum into 6 equal bands)
    band_e   = {f'band_energy_{i}': np.sum(psd[i*(n//12):(i+1)*(n//12)]) for i in range(6)}
    feats    = {
        'spectral_centroid': centroid,
        'spectral_spread':   spread,
        'spectral_rolloff':  rolloff,
        'spectral_flatness': flatness,
    }
    feats.update(band_e)
    return feats

def extract_wavelet_features(window, wavelet='db4', level=3):
    """8 normalized WPD energy bands (Level-3 db4)."""
    wp    = pywt.WaveletPacket(data=window, wavelet=wavelet, mode='symmetric', maxlevel=level)
    nodes = [node.path for node in wp.get_level(level, 'freq')]
    energies    = [np.sum(np.square(wp[n].data)) for n in nodes]
    total_energy = sum(energies)
    return {f'wpd_energy_node_{i}': e / total_energy if total_energy > 0 else 0
            for i, e in enumerate(energies)}

def extract_features(window):
    """Combined feature extraction → matches scripts/feature_engineering.py."""
    f = {**extract_time_features(window)}
    f.update(extract_freq_features(window))
    f.update(extract_wavelet_features(window))
    return f

## 2. Processing Pipeline (Filter + Sliding Window)

In [ ]:
all_features = []

for category in os.listdir(INTERIM_DIR):
    cat_path = os.path.join(INTERIM_DIR, category)
    if not os.path.isdir(cat_path):
        continue
    if category not in ALLOWED_CATEGORIES:
        print(f"Skipping: {category}")
        continue

    files = [f for f in os.listdir(cat_path) if f.endswith('.parquet')]
    for filename in tqdm(files, desc=category):
        df = pd.read_parquet(os.path.join(cat_path, filename))
        if 'vibration_de' not in df.columns:
            continue

        signal = df['vibration_de'].values
        meta   = {
            'fault_type':     df['fault_type'].iloc[0],
            'fault_diameter': df['fault_diameter'].iloc[0],
            'load':           df['load'].iloc[0],
            'rpm':            df['rpm'].iloc[0],
        }

        # Sliding window with 75% overlap
        start = 0
        while start + WINDOW_SIZE <= len(signal):
            window = signal[start:start + WINDOW_SIZE]
            row = {**meta, **extract_features(window)}
            all_features.append(row)
            start += STRIDE

final_df = pd.DataFrame(all_features)

# Standardize OR subtypes → OR
final_df['fault_class'] = final_df['fault_type'].apply(lambda x: 'OR' if str(x).startswith('OR') else x)
# Label encode alphabetically: B=0, IR=1, Normal=2, OR=3
final_df['label'] = pd.Categorical(final_df['fault_class']).codes

print("\nRaw class distribution:")
print(final_df.groupby(['fault_class', 'label']).size())

## 3. Class Balancing → 4,000 samples per class

In [ ]:
TARGET = 4000
parts  = []

for cls in sorted(final_df['label'].unique()):
    chunk = final_df[final_df['label'] == cls]
    label_name = chunk['fault_class'].iloc[0]
    if len(chunk) > TARGET:
        chunk = chunk.sample(n=TARGET, random_state=42)
        print(f"[{label_name}] undersampled → {TARGET}")
    else:
        chunk = chunk.sample(n=TARGET, replace=True, random_state=42)
        print(f"[{label_name}] upsampled  → {TARGET}")
    parts.append(chunk)

balanced_df = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)

output_path = os.path.join(PROCESSED_DATA_PATH, "cwru_features.parquet")
balanced_df.to_parquet(output_path, index=False)
balanced_df.head(100).to_csv(output_path.replace('.parquet', '_sample.csv'), index=False)
print(f"\nSaved: {output_path} | shape={balanced_df.shape}")
print(balanced_df.groupby(['fault_class', 'label']).size())